In [23]:
from tavily import TavilyClient
from langchain_core.tools import tool
from dotenv import load_dotenv
import os

load_dotenv()

client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


def web_search_impl(query: str) -> str:
    """Search the web for recent and factual information (callable helper).

    This implementation is usable directly in scripts/tests. The agent-facing
    StructuredTool is exposed as `web_search_tool` and kept for compatibility.
    """
    try:
        response = client.search(
            query=query,
            search_depth="advanced",
            max_results=5,
            include_answer=True,
            include_raw_content=False,
            include_images=False,
        )

        answer = response.get("answer", "")
        results = response.get("results", [])

        formatted = f"Answer:\n{answer}\n\nSources:\n"

        for i, result in enumerate(results, 1):
            formatted += (
                f"\n{i}. {result.get('title','')}\n"
                f"URL: {result.get('url','')}\n"
                f"{result.get('content','')}\n"
            )

        return formatted

    except Exception as e:
        return f"Search error: {str(e)}"


# Agent-facing StructuredTool (kept for imports that expect `web_search`)
web_search_tool = tool(web_search_impl)
# Backwards compatibility: export name `web_search` as the StructuredTool
web_search = web_search_tool

print('tavily import cell loaded')

result = web_search.invoke({"query": "What is LangGraph?"})
print(result)

tavily import cell loaded
Answer:
LangGraph is an open-source framework for building stateful, multi-step AI agent workflows using directed graphs. It manages complex interactions and maintains state across steps. It's used for applications like customer support and chatbots.

Sources:

1. What is LangGraph? | Decagon glossary | Decagon
URL: https://decagon.ai/glossary/what-is-langgraph
Introducing Duet Autopilot.

Learn more

Sign in

Get a demo

Glossary

# LangGraph

LangGraph is an open-source orchestration framework for building stateful, multi-step AI agent workflows as directed graphs, where nodes represent actions or model calls and edges define the control flow between them. [...] ## How LangGraph works

LangGraph represents an agent workflow as a stateful directed graph. Nodes are Python functions, model calls, or tool invocations. Edges are either unconditional, always proceeding to the next node, or conditional, routing to different nodes based on the output of the precedin

In [24]:

import os
import re
import certifi
import requests
import airportsdata
import pycountry

from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv()

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

API_KEY = os.getenv("AVIATIONSTACK_API_KEY")
BASE_URL = "https://api.aviationstack.com/v1/flights"

AIRPORTS = airportsdata.load("IATA")

COUNTRY_ALIASES = {
    "usa":"US","us":"US","united states":"US","united states of america":"US",
    "uk":"GB","united kingdom":"GB","great britain":"GB","britain":"GB","england":"GB",
    "pakistan":"PK","pak":"PK","pk":"PK",
    "india":"IN","ind":"IN",
    "uae":"AE","united arab emirates":"AE","emirates":"AE",
}

def clean_text(text:str)->str:
    text=text.lower().strip()
    text=re.sub(r"[^a-z0-9\s]"," ",text)
    text=re.sub(r"\s+"," ",text)
    stop={"flight","flights","ticket","tickets","trip","travel","plan","planning","complete",
          "day","days","including","hotel","hotels","sightseeing","under","budget",
          "info","information","for","to","from","of","the","a","an"}
    return " ".join(w for w in text.split() if w not in stop)

def country_name_to_code(text:str):
    text=text.lower().strip()
    try:
        return pycountry.countries.lookup(text).alpha_2
    except LookupError:
        pass
    for c in pycountry.countries:
        if c.name.lower() in text:
            return c.alpha_2
    for a,code in COUNTRY_ALIASES.items():
        if a in text:
            return code
    return None

def airport_country_matches(airport:dict,country_code:str)->bool:
    country=airport.get("country","")
    if country.upper()==country_code:
        return True
    obj=pycountry.countries.get(alpha_2=country_code)
    return bool(obj and country.lower()==obj.name.lower())

def get_best_airport_for_country(country_code:str):
    best=None
    score=-1
    for iata,airport in AIRPORTS.items():
        if airport_country_matches(airport,country_code):
            s=0
            name=airport.get("name","").lower()
            if "international" in name: s+=50
            if "intl" in name: s+=40
            if s>score:
                score=s
                best=iata
    return best

def resolve_location_to_iata(location:str):
    if not location:
        return None
    raw=location.strip()
    if re.fullmatch(r"[A-Za-z]{3}",raw):
        return raw.upper() if raw.upper() in AIRPORTS else None
    cc=country_name_to_code(raw)
    if cc:
        return get_best_airport_for_country(cc)
    target=clean_text(raw)
    for iata,a in AIRPORTS.items():
        if a.get("city","").lower()==target:
            return iata
    return None

def parse_route(query:str):
    m=re.search(r"from (.+?) to (.+)",query,re.I)
    if m:
        return resolve_location_to_iata(m.group(1)),resolve_location_to_iata(m.group(2))
    codes=re.findall(r"\b[A-Z]{3}\b",query)
    if len(codes)>=2:
        return codes[0],codes[1]
    return None,None

def format_flight(flight:dict)->str:
    dep=flight.get("departure",{})
    arr=flight.get("arrival",{})
    return f"""Airline: {flight.get('airline',{}).get('name')}
Flight: {flight.get('flight',{}).get('iata')}
Status: {flight.get('flight_status')}

Departure: {dep.get('airport')} ({dep.get('iata')})
Scheduled: {dep.get('scheduled')}

Arrival: {arr.get('airport')} ({arr.get('iata')})
Scheduled: {arr.get('scheduled')}
"""

@tool
def search_flights(query:str)->str:
    """Search flights by natural language route."""
    dep,arr=parse_route(query)
    params={"access_key":API_KEY}
    if dep: params["dep_iata"]=dep
    if arr: params["arr_iata"]=arr
    r=requests.get(BASE_URL,params=params,timeout=30)
    r.raise_for_status()
    data=r.json().get("data",[])
    if not data:
        return "No flights found."
    return "\n\n".join(format_flight(f) for f in data[:5])

print(search_flights.invoke({
    "query": "Flights from Lahore to Dubai"
}))


Airline: airblue
Flight: PA410
Status: scheduled

Departure: Alama Iqbal International (LHE)
Scheduled: 2026-08-08T04:50:00+00:00

Arrival: Dubai (DXB)
Scheduled: 2026-08-08T07:20:00+00:00


Airline: Pakistan International Airlines
Flight: PK203
Status: scheduled

Departure: Alama Iqbal International (LHE)
Scheduled: 2026-08-08T00:50:00+00:00

Arrival: Dubai (DXB)
Scheduled: 2026-08-08T03:15:00+00:00


Airline: Ethiopian Airlines
Flight: ET4359
Status: scheduled

Departure: Alama Iqbal International (LHE)
Scheduled: 2026-08-08T00:50:00+00:00

Arrival: Dubai (DXB)
Scheduled: 2026-08-08T03:15:00+00:00


Airline: airblue
Flight: PA416
Status: active

Departure: Alama Iqbal International (LHE)
Scheduled: 2026-08-07T15:50:00+00:00

Arrival: Dubai (DXB)
Scheduled: 2026-08-07T18:20:00+00:00


Airline: Emirates
Flight: EK625
Status: landed

Departure: Alama Iqbal International (LHE)
Scheduled: 2026-08-07T11:55:00+00:00

Arrival: Dubai (DXB)
Scheduled: 2026-08-07T14:10:00+00:00



In [ ]:
import os
import certifi
from dotenv import load_dotenv

load_dotenv()

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# ==========================
# LangGraph Imports
# ==========================

from langgraph.graph import MessagesState, StateGraph, START, END

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage,
)

from langchain_groq import ChatGroq

from app.agents.tools.tavily_tool import web_search
from app.agents.tools.flight_tool import search_flights

# ==========================
# LLM
# ==========================

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)

# ==========================
# State
# ==========================

class TravelState(MessagesState):
    user_query="lahore to dubai"
    flight_result: str
    hotel_result: str
    itinerary: str
    llms_calls: int

# ==========================
# Flight Agent
# ==========================

def flight_agent(state: TravelState):
    query = state["user_query"]

    flight_data = search_flights.invoke({
        "query": query
    })
    

    return {
        "flight_result": flight_data,
        "llms_calls": state.get("llms_calls", 0) + 1,
    }

initial_state = {
    "user_query": "Lahore to dubai",
    "messages": [],
    "llms_calls": 0,
}

result = flight_agent(initial_state)

print(result)

{'flight_result': 'Airline: empty\nFlight: None\nStatus: active\n\nDeparture: Charleville (CTL)\nScheduled: 2026-08-09T10:30:00+00:00\n\nArrival: Longreach (LRE)\nScheduled: 2026-08-07T13:00:00+00:00\n\n\nAirline: Wings Air\nFlight: IW1887\nStatus: scheduled\n\nDeparture: Waingapu (WGP)\nScheduled: 2026-08-08T07:45:00+00:00\n\nArrival: Lombok International (LOP)\nScheduled: 2026-08-08T09:05:00+00:00\n\n\nAirline: Singapore Airlines\nFlight: SQ8638\nStatus: scheduled\n\nDeparture: Singapore Changi (SIN)\nScheduled: 2026-08-08T02:45:00+00:00\n\nArrival: Vienna International (VIE)\nScheduled: 2026-08-08T08:55:00+00:00\n\n\nAirline: Virgin Australia\nFlight: VA5602\nStatus: active\n\nDeparture: Singapore Changi (SIN)\nScheduled: 2026-08-08T02:35:00+00:00\n\nArrival: Indira Gandhi International (DEL)\nScheduled: 2026-08-08T05:40:00+00:00\n\n\nAirline: Air New Zealand\nFlight: NZ3382\nStatus: active\n\nDeparture: Singapore Changi (SIN)\nScheduled: 2026-08-08T02:35:00+00:00\n\nArrival: Indira

In [2]:
import os
import certifi
from dotenv import load_dotenv

load_dotenv()

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# ==========================
# LangGraph Imports
# ==========================

from langgraph.graph import MessagesState, StateGraph, START, END

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage,
)

from langchain_groq import ChatGroq

from app.agents.tools.tavily_tool import web_search
from app.agents.tools.flight_tool import search_flights

# ==========================
# LLM
# ==========================

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
)

# ==========================
# State
# ==========================

class TravelState(MessagesState):
    user_query: str
    flight_result: str
    hotel_result: str
    itinerary: str
    llm_calls: int

# ==========================
# Flight Agent
# ==========================

def flight_agent(state: TravelState):
    query= state["user_query"]
   
    flight_data = search_flights.invoke({
        "query": query
    })
    
    return {
        "flight_results":flight_data,
        "messages":[
            AIMessage(content="Flight results fetched")
        ],
        "llm_calls":state.get("llm_calls",0)+ 1
    }
   
# ==========================
# Hotel Agent
# ==========================

def hotel_agent(state: TravelState):
    query = f"Best hotels for {state['user_query']}"

    hotel_results = web_search.invoke({
        "query": query
    })

    return {
        "hotel_result": hotel_results,
        "messages": state["messages"] + [
            AIMessage(content="Hotel results fetched successfully.")
        ],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }
# ==========================
# Itinerary Agent
# ==========================

def itinerary_agent(state: TravelState):
    user_query = state["user_query"]
    flight_result = state.get("flight_result", "No flight information found.")
    hotel_result = state.get("hotel_result", "No hotel information found.")

    prompt = f"""
You are an expert travel planner.

Create a complete travel itinerary using the following information.

User Request:
{user_query}

Flight Information:
{flight_result}

Hotel Information:
{hotel_result}

Generate a well-formatted itinerary with:

1. Trip Summary
2. Recommended Flight
3. Recommended Hotel
4. Day-by-Day Plan
5. Estimated Budget (if possible)
6. Travel Tips

Return the response in Markdown.
"""

    response = llm.invoke([
        SystemMessage(content="You are an expert travel planner."),
        HumanMessage(content=prompt),
    ])

    return {
        "itinerary": response.content,
        "messages": state["messages"] + [
            AIMessage(content="Travel itinerary created successfully.")
        ],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }


# ==========================
# Final Result Agent
# ==========================

def final_result_agent(state: TravelState):
    prompt = f"""
You are an AI Travel Assistant.

Prepare a final response for the user using the information below.

User Request:
{state["user_query"]}

Flight Details:
{state.get("flight_result", "Not available")}

Hotel Details:
{state.get("hotel_result", "Not available")}

Travel Itinerary:
{state.get("itinerary", "Not available")}

Instructions:
- Write in a friendly and professional tone.
- Use Markdown formatting.
- Include:
  1. Flight Recommendation
  2. Hotel Recommendation
  3. Complete Itinerary
  4. Important Travel Tips
- End with: "Have a safe and enjoyable trip! ✈️"
"""

    response = llm.invoke([
        SystemMessage(content="You are an expert AI Travel Assistant."),
        HumanMessage(content=prompt)
    ])

    return {
        "messages": state["messages"] + [
            AIMessage(content=response.content)
        ],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }
    # Create Graph
builder = StateGraph(TravelState)

# ==========================
# Add Nodes
# ==========================

builder.add_node("flight_agent", flight_agent)
builder.add_node("hotel_agent", hotel_agent)
builder.add_node("itinerary_agent", itinerary_agent)
builder.add_node("final_result_agent", final_result_agent)

# ==========================
# Add Edges
# ==========================

builder.add_edge(START, "flight_agent")
builder.add_edge("flight_agent", "hotel_agent")
builder.add_edge("hotel_agent", "itinerary_agent")
builder.add_edge("itinerary_agent", "final_result_agent")
builder.add_edge("final_result_agent", END)

# ==========================
# Compile Graph
# ==========================

graph = builder.compile()

query1="lahore to dubai"
state = {
        "user_query": query1,
        "messages": [
            HumanMessage(content=query1)
        ],
        "llm_calls": 0,
    }

result = graph.invoke(state)
result

{'messages': [HumanMessage(content='lahore to dubai', additional_kwargs={}, response_metadata={}, id='e1da379d-7d91-41d0-8b4e-2fcf05639e3f'),
  AIMessage(content='Flight results fetched', additional_kwargs={}, response_metadata={}, id='64b07448-e93b-457e-8ff6-755bab5c959c', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='Hotel results fetched successfully.', additional_kwargs={}, response_metadata={}, id='5c020a5f-24a9-44ea-8513-b34056a71b87', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='Travel itinerary created successfully.', additional_kwargs={}, response_metadata={}, id='dd5a0e7e-e8d4-44b1-b7d2-e0bf37945a73', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content="### Trip Summary\nThis trip itinerary is designed for travel from Lahore to Dubai, with a focus on convenient flights, comfortable accommodations, and a well-planned daily schedule. Given the lack of specific flight information, we will outline a general approach to booking flights and recom

In [ ]:
def flight_agent(state: TravelState):
    user_query = state["user_query"]

    # Use LLM to extract a clean flight query
    response = llm.invoke([
        SystemMessage(
            content="""
You are a flight search query extractor.

Extract ONLY the flight route from the user's travel request.

Return the route in this exact format:
ORIGIN to DESTINATION

Examples:

User: Plan a trip from Lahore to Dubai and find hotels
Output: Lahore to Dubai

User: I want to travel from Karachi to London
Output: Karachi to London

User: Book me a flight from Islamabad to Istanbul next week
Output: Islamabad to Istanbul

Do not include:
- hotels
- sightseeing
- dates
- budget
- airlines
- extra explanation

Return ONLY the origin and destination.
"""
        ),
        HumanMessage(content=user_query)
    ])

    flight_query = response.content.strip()

    # Send extracted query to flight tool
    flight_data = search_flights.invoke({
        "query": flight_query
    })

    return {
        "flight_result": flight_data,
        "messages": [
            AIMessage(
                content=f"Flight search completed for {flight_query}"
            )
        ],
        "llm_calls": state.get("llm_calls", 0) + 1,
    }
    
    